In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split

# load raw data
df = pd.read_csv('../data/raw/Suicide_Detection.csv')
df = df.drop(columns=['Unnamed: 0'])

# encode labels as integers
df['label'] = (df['class'] == 'suicide').astype(int)

print(f"Label encoding: suicide=1, non-suicide=0")
print(df[['class', 'label']].head())


Label encoding: suicide=1, non-suicide=0
         class  label
0      suicide      1
1  non-suicide      0
2  non-suicide      0
3      suicide      1
4      suicide      1


In [2]:
def clean_text(text):
    # lowercase
    text = text.lower()
    # remove urls
    text = re.sub(r'http\S+|www\S+', '', text)
    # remove reddit-style formatting (e.g. r/subreddit, u/username)
    text = re.sub(r'r/\w+|u/\w+', '', text)
    # remove special characters and numbers, keep letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)
    # remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# apply cleaning
df['clean_text'] = df['text'].apply(clean_text)

# verify
print("Original:")
print(df['text'].iloc[0][:200])
print("\nCleaned:")
print(df['clean_text'].iloc[0][:200])


Original:
Ex Wife Threatening SuicideRecently I left my wife for good because she has cheated on me twice and lied to me so much that I have decided to refuse to go back to her. As of a few days ago, she began 

Cleaned:
ex wife threatening suiciderecently i left my wife for good because she has cheated on me twice and lied to me so much that i have decided to refuse to go back to her as of a few days ago she began th


In [3]:
# train/val/test split (80/10/10)
train_df, temp_df = train_test_split(df[['clean_text', 'label']], test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"\nTrain label balance:\n{train_df['label'].value_counts()}")

# save to processed
train_df.to_csv('../data/processed/train.csv', index=False)
val_df.to_csv('../data/processed/val.csv', index=False)
test_df.to_csv('../data/processed/test.csv', index=False)

print("\nSaved to data/processed/")

Train: 185659 | Val: 23207 | Test: 23208

Train label balance:
label
1    92830
0    92829
Name: count, dtype: int64

Saved to data/processed/
